# POC_DDM_final (6-chip) dataset profile

Builds the target x concentration curve-count profile for the six chips in the
`final_6_chip_clean_nn` LOFO group, including the positive-control (PC) and
negative-control (NC) wells:

```
D20260806_E00_C00_F4500KHz_U_DDM_01_06   ->  DDM_01
D20260807_E00_C00_F4500KHz_U_DDM_02_07   ->  DDM_02
D20260808_E00_C00_F4500KHz_U_DDM_03_01   ->  DDM_03
D20260810_E00_C00_F4500KHz_U_DDM_04_01   ->  DDM_04
D20260825_E00_C00_F4500KHz_U_DDM_05_01   ->  DDM_05
D20260825_E00_C00_F4500KHz_U_DDM_06_02   ->  DDM_06
```

Counts are read directly from each chip's `curve_for_training.joblib` (the same file `01_curve_preprocessing_v6.py` produces and every downstream script consumes), so the numbers below are the actual per-well active-pixel curve counts, not estimates.

Three things worth knowing before reading the output:
- **Positive control (well 8) is stored separately**, under the `pc_wells` key rather than in the main `Y_well`/`dataset` arrays -- `01_curve_preprocessing_v6.py --drop_pc` snapshots it out before the main NC-subtraction step. It's read from there and added back in below as its own row.
- **Not every nominal well has data.** `config.LABEL_MAPPINGS`/`config.CONC_MAPPINGS` assign a target+concentration to wells 0-9 for every chip, but a well can end up with zero active-pixel curves (e.g. DDM_01's second Hadv well, index 7, has none). The table reports what's actually in the data, so a target/concentration cell can legitimately be 0.
- **The LaTeX table strikes through cells excluded from LOFO training** for that chip, per `config.LOFO_EXCLUDE_WELL_MAPPING['final_6_chip_clean_nn']` -- PC/NC are excluded for every chip (not LOFO-trainable targets), plus a few chip-specific extra wells. Requires `\usepackage{soul}` for `\sout{}` in the LaTeX build.


In [1]:
import os
import sys
import re
import collections

import joblib
import numpy as np
import pandas as pd

# VSCode's Jupyter kernel doesn't reliably start with CWD = this notebook's own
# directory (and a stray chdir elsewhere in the session can also leave it wrong) --
# anchor off the notebook's own absolute path instead of trusting os.getcwd().
try:
    _nb = globals().get('__vsc_ipynb_file__')
    if _nb:
        os.chdir(os.path.dirname(_nb))
except Exception:
    pass

_MAIN_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, _MAIN_DIR)
sys.path.insert(0, os.path.join(_MAIN_DIR, "utils"))
sys.path.insert(0, os.path.join(_MAIN_DIR, "utils", "model_training"))

import config

GROUP_NAME = "final_6_chip_clean_nn"
EXP_FOLDER = "/vol/bitbucket/gk225/POC_DDM_datasets/POC_DDM_final"
DATASETS = config.CROSS_DATASET_GROUPS[GROUP_NAME]
PC_WELL_IDX = 8  # every chip's LABEL_MAPPINGS assigns well 8 to PC

# Paper-facing target names, matching the methodology chapter's own wording
# ("five targets (IAV, IBV, KP, SARS-CoV-2 and hAdV)") rather than the code's
# internal shorthand (Kp/Cov/Hadv).
TARGET_DISPLAY_NAME = {
    "IAV": "IAV", "IBV": "IBV", "Kp": "KP", "Cov": "SARS-CoV-2", "Hadv": "hAdV",
}
TARGET_ORDER = ["IAV", "IBV", "KP", "SARS-CoV-2", "hAdV"]
CONC_ORDER = [10_000, 100_000, 1_000_000]
CONC_DISPLAY = {10_000: "$10^4$", 100_000: "$10^5$", 1_000_000: "$10^6$"}


def chip_label(dataset_name):
    """'D20260806_E00_C00_F4500KHz_U_DDM_01_06' -> 'DDM_01'"""
    m = re.search(r"DDM_(\d+)", dataset_name)
    return f"DDM_{m.group(1)}"



[*] SUCCESS: TensorFlow is utilizing the GPU -> /physical_device:GPU:0



## Load per-well curve counts

For each chip: count curves per well index in the main `Y_well` array (targets + `NC-ALL`), skipping well 8 (PC) there since it's always empty post-`--drop_pc`; then add the real PC count from the separately-stored `pc_wells` snapshot.

In [2]:
def load_well_counts(dataset_name):
    """Returns a list of (well_idx, label, concentration, n_curves) for one chip:
    the main Y_well wells (targets + NC-ALL) plus the separately-stored PC well."""
    data_path = os.path.join(EXP_FOLDER, dataset_name, config.TRAINING_DATA_PATH)
    d = joblib.load(data_path)

    label_map = config.LABEL_MAPPINGS.get(dataset_name, {})
    conc_map = config.CONC_MAPPINGS.get(dataset_name, {})

    Y_well = np.asarray(d["Y_well"])
    counts = collections.Counter(Y_well.tolist())

    rows = []
    for well_idx, label in label_map.items():
        if well_idx == PC_WELL_IDX:
            continue  # real PC count comes from pc_wells below, not main Y_well
        n = counts.get(well_idx, 0)
        rows.append((well_idx, label, conc_map.get(well_idx), n))

    # Positive control: dropped from Y_well by --drop_pc, snapshotted separately.
    pc = d.get("pc_wells")
    if pc and "Y_well" in pc:
        n_pc = len(pc["Y_well"])
    elif pc and "curves" in pc:
        first_variant = next(iter(pc["curves"].values()))
        n_pc = np.asarray(first_variant).shape[0]
    else:
        n_pc = 0
    rows.append((PC_WELL_IDX, "PC", 0, n_pc))

    return rows


records = []
for name in DATASETS:
    for well_idx, label, conc, n in load_well_counts(name):
        records.append({
            "chip": chip_label(name),
            "well": well_idx,
            "label": label,
            "concentration": conc,
            "n_curves": n,
        })

well_df = pd.DataFrame.from_records(records)
well_df

,chip,well,label,concentration,n_curves
0,DDM_01,0,IAV,1000000,1970
1,DDM_01,1,IAV,100000,1949
2,DDM_01,2,IAV,10000,1858
3,DDM_01,3,IBV,10000,1907
4,DDM_01,4,Kp,100000,1921
5,DDM_01,5,Cov,1000000,1848
6,DDM_01,6,Hadv,1000000,1919
7,DDM_01,7,Hadv,100000,0
8,DDM_01,9,NC-ALL,0,1135
9,DDM_01,8,PC,0,791


## Pivot into the profile table

Rows = chip. Columns = each target split by concentration, plus **PC**, **NC**, and a **Total** column (sum of every well on that chip, including PC and NC). This wide, chip-per-row form is transposed below into the layout actually used for the LaTeX table.

In [3]:
def build_profile_table(well_df):
    df = well_df.copy()
    df["target"] = df["label"].map(TARGET_DISPLAY_NAME)

    target_rows = df[df["target"].notna()]
    wide = target_rows.pivot_table(
        index="chip", columns=["target", "concentration"],
        values="n_curves", aggfunc="sum", fill_value=0,
    )

    col_order = pd.MultiIndex.from_product([TARGET_ORDER, CONC_ORDER], names=["target", "concentration"])
    wide = wide.reindex(columns=col_order, fill_value=0)

    pc = df[df["label"] == "PC"].groupby("chip")["n_curves"].sum()
    nc = df[df["label"] == "NC-ALL"].groupby("chip")["n_curves"].sum()

    wide[("PC", "")] = pc.reindex(wide.index, fill_value=0)
    wide[("NC", "")] = nc.reindex(wide.index, fill_value=0)
    wide[("Total", "")] = wide.sum(axis=1)

    wide = wide.reindex(index=[chip_label(n) for n in DATASETS])
    wide.columns = pd.MultiIndex.from_tuples(wide.columns, names=["target", "concentration"])
    return wide


profile = build_profile_table(well_df)
profile

target          IAV                  IBV                   KP                 \
concentration 10000 100000 1000000 10000 100000 1000000 10000 100000 1000000   
chip                                                                           
DDM_01         1858   1949    1970  1907      0       0     0   1921       0   
DDM_02            0   1855    1816  1469   1799    1047  1911      0       0   
DDM_03         1786      0       0     0   1852    1903  1960   1663    1964   
DDM_04            0      0    1920  1921      0       0     0   1884    1950   
DDM_05            0   1955       0     0      0    1919  1943      0       0   
DDM_06         1890      0       0  1827   1936       0     0      0    1864   

target        SARS-CoV-2                 hAdV                   PC    NC  \
concentration      10000 100000 1000000 10000 100000 1000000               
chip                                                                       
DDM_01                 0      0    1848     0      0    1919   791  1135   
DDM_02                 0   1851       0  1894      0       0  1814  1778   
DDM_03              1928      0       0     0      0    1828  1755  1642   
DDM_04              1920   1913    1808     0   1943       0  1861  1315   
DDM_05                 0   1921    1884  1903   1945    1979  1896  1901   
DDM_06              1941   1916       0  3787      0       0  1715  1810   

target         Total  
concentration         
chip                  
DDM_01         15298  
DDM_02         17234  
DDM_03         18281  
DDM_04         18435  
DDM_05         19246  
DDM_06         18686

## Transposed view (target x concentration as rows, chip as columns)

With only 4 chips but 15 target/concentration combinations, a tall table (rows = target x concentration, plus PC/NC/Total; columns = chip) reads far better than the 18-column wide form above, and is what the LaTeX export below produces.

In [4]:
row_order = pd.MultiIndex.from_tuples(
    [(t, c) for t in TARGET_ORDER for c in CONC_ORDER] + [("PC", ""), ("NC", ""), ("Total", "")],
    names=["target", "concentration"],
)
profile_T = profile.T.reindex(row_order)

display_index = pd.MultiIndex.from_tuples(
    [(t, CONC_DISPLAY.get(c, c) if c != "" else "") for (t, c) in profile_T.index],
    names=["target", "concentration"],
)
profile_T_display = profile_T.copy()
profile_T_display.index = display_index
profile_T_display.style.format("{:,}")

## LaTeX table (transposed)

Booktabs style, matching `tab:notation` / `tab:lab_datasets` in the methodology chapter. Rows = target x concentration (`\multirow` groups each target's three concentrations) plus dedicated **PC** and **NC** rows and a **Total** row; columns = chip. Ready to drop into `03_methodology.tex` under \S\ref{app:chip_mapping} or the Lacewing eLAMP Platform subsection.

In [5]:
def build_lofo_excluded_cells(group_name):
    """{(chip_label, row_label, concentration)} for every well
    config.LOFO_EXCLUDE_WELL_MAPPING drops for this LOFO group -- PC/NC use
    concentration="" to match how build_profile_table keys those two rows."""
    excluded = set()
    mapping = config.LOFO_EXCLUDE_WELL_MAPPING.get(group_name) or {}
    for dataset_name, wells in mapping.items():
        label_map = config.LABEL_MAPPINGS.get(dataset_name, {})
        conc_map = config.CONC_MAPPINGS.get(dataset_name, {})
        chip = chip_label(dataset_name)
        for w in wells:
            raw_label = label_map.get(w)
            if raw_label is None:
                continue
            if raw_label == "PC":
                excluded.add((chip, "PC", ""))
            elif raw_label == "NC-ALL":
                excluded.add((chip, "NC", ""))
            else:
                target = TARGET_DISPLAY_NAME.get(raw_label, raw_label)
                excluded.add((chip, target, conc_map.get(w)))
    return excluded


_CHIP_DISPLAY_RENAME = ("DDM_0", "Chip 0")  # matches 4_visualise_recon_layer.ipynb's chip_title()


def chip_display(chip):
    return chip.replace(*_CHIP_DISPLAY_RENAME)


_CONTROL_DISPLAY = {"PC": "Positive Control", "NC": "Negative Control"}


def to_latex_profile_table_transposed(profile, caption, label, excluded_cells=None):
    """excluded_cells=None (or empty) -> plain table; pass build_lofo_excluded_cells(...)
    for a 2nd version with those cells \\sout{}-struck (LOFO-excluded)."""
    excluded_cells = excluded_cells or set()
    chips = list(profile.index)
    n_chips = len(chips)

    def fmt(chip, row_label, conc, v):
        s = f"{v:,}" if v else "--"
        return f"\\sout{{{s}}}" if (chip, row_label, conc) in excluded_cells else s

    col_spec = "ll" + "r" * n_chips
    lines = []
    lines.append("\\begin{table}[htbp]")
    lines.append("    \\centering")
    lines.append(f"    \\caption{{{caption}}}")
    lines.append(f"    \\label{{{label}}}")
    lines.append("    \\small")
    lines.append(f"    \\begin{{tabular}}{{@{{}}{col_spec}@{{}}}}")
    lines.append("    \\toprule")

    header = ["\\textbf{Target}", "\\textbf{Conc.}"] + [f"\\textbf{{{chip_display(c)}}}" for c in chips]
    lines.append("    " + " & ".join(header) + " \\\\")
    lines.append("    \\midrule")

    def get(chip, col):
        return profile.loc[chip, col] if col in profile.columns else 0

    for t in TARGET_ORDER:
        for i, c in enumerate(CONC_ORDER):
            target_cell = f"\\multirow{{{len(CONC_ORDER)}}}{{*}}{{{t}}}" if i == 0 else ""
            cells = [target_cell, CONC_DISPLAY[c]]
            for chip in chips:
                v = get(chip, (t, c))
                cells.append(fmt(chip, t, c, v))
            lines.append("    " + " & ".join(cells) + " \\\\")
        lines.append("    \\midrule")

    # Positive and negative control rows
    for extra in ["PC", "NC"]:
        cells = [f"\\multicolumn{{2}}{{l}}{{{_CONTROL_DISPLAY[extra]}}}"]
        for chip in chips:
            v = get(chip, (extra, ''))
            cells.append(fmt(chip, extra, '', v))
        lines.append("    " + " & ".join(cells) + " \\\\")

    lines.append("    \\midrule")
    cells = ["\\multicolumn{2}{l}{\\textbf{Total}}"]
    for chip in chips:
        cells.append(f"{get(chip, ('Total', '')):,}")
    lines.append("    " + " & ".join(cells) + " \\\\")

    lines.append("    \\bottomrule")
    lines.append("    \\end{tabular}")
    lines.append("\\end{table}")
    return "\n".join(lines)


# Version 1: plain, no strikethrough -- matches the reference style/wording exactly.
latex_table_plain = to_latex_profile_table_transposed(
    profile,
    caption="Per-well active-pixel curve counts across the six Lacewing eLAMP chips. "
            "The data is grouped by target and concentration alongside the respective "
            "positive and negative control wells.",
    label="tab:final_6chip_profile",
)
print(latex_table_plain)


\begin{table}[htbp]
    \centering
    \caption{Per-well active-pixel curve counts across the six Lacewing eLAMP chips in the final\_6\_chip\_clean\_nn LOFO group, by target and concentration (copies/reaction), including each chip's positive (PC) and negative (NC) control wells. Struck-through values are excluded from LOFO training for that chip.}
    \label{tab:final_6chip_profile}
    \small
    \begin{tabular}{@{}llrrrrrr@{}}
    \toprule
    \textbf{Target} & \textbf{Conc.} & \textbf{DDM_01} & \textbf{DDM_02} & \textbf{DDM_03} & \textbf{DDM_04} & \textbf{DDM_05} & \textbf{DDM_06} \\
    \midrule
    \multirow{3}{*}{IAV} & $10^4$ & 1,858 & -- & 1,786 & -- & -- & 1,890 \\
     & $10^5$ & 1,949 & 1,855 & -- & -- & 1,955 & -- \\
     & $10^6$ & 1,970 & 1,816 & -- & 1,920 & -- & -- \\
    \midrule
    \multirow{3}{*}{IBV} & $10^4$ & 1,907 & 1,469 & -- & 1,921 & -- & \sout{1,827} \\
     & $10^5$ & -- & 1,799 & 1,852 & -- & -- & 1,936 \\
     & $10^6$ & -- & 1,047 & 1,903 & -- & 1,919 & 

In [12]:
# Version 2: same style, with cells config.LOFO_EXCLUDE_WELL_MAPPING[GROUP_NAME]
# drops for LOFO training struck through (\\sout{}).
excluded_cells = build_lofo_excluded_cells(GROUP_NAME)
latex_table_lofo = to_latex_profile_table_transposed(
    profile,
    caption="Per-well active-pixel curve counts across the six Lacewing eLAMP chips. "
            "The data is grouped by target and concentration alongside the respective "
            "positive and negative control wells. Struck-through values are excluded "
            f"from LOFO training for the {GROUP_NAME.replace(chr(95), chr(92) + chr(95))} group.",
    label="tab:final_6chip_profile_lofo",
    excluded_cells=excluded_cells,
)
print(latex_table_lofo)


\begin{table}[htbp]
    \centering
    \caption{Per-well active-pixel curve counts across the six Lacewing eLAMP chips. The data is grouped by target and concentration alongside the respective positive and negative control wells. Struck-through values are excluded from LOFO training for the final\_6\_chip\_clean\_nn group.}
    \label{tab:final_6chip_profile_lofo}
    \small
    \begin{tabular}{@{}llrrrrrr@{}}
    \toprule
    \textbf{Target} & \textbf{Conc.} & \textbf{DDM_01} & \textbf{DDM_02} & \textbf{DDM_03} & \textbf{DDM_04} & \textbf{DDM_05} & \textbf{DDM_06} \\
    \midrule
    \multirow{3}{*}{IAV} & $10^4$ & 1,858 & -- & 1,786 & -- & -- & 1,890 \\
     & $10^5$ & 1,949 & 1,855 & -- & -- & 1,955 & -- \\
     & $10^6$ & 1,970 & 1,816 & -- & 1,920 & -- & -- \\
    \midrule
    \multirow{3}{*}{IBV} & $10^4$ & 1,907 & 1,469 & -- & 1,921 & -- & \sout{1,827} \\
     & $10^5$ & -- & 1,799 & 1,852 & -- & -- & 1,936 \\
     & $10^6$ & -- & 1,047 & 1,903 & -- & 1,919 & -- \\
    \midrule


In [6]:
# out_path = os.path.join(_MAIN_DIR, "notebooks", "final_6chip_dataset_profile_table.tex")
# with open(out_path, "w") as f:
#     f.write(latex_table + "\n")
# print(f"Saved -> {out_path}")



## Active vs. non-active pixel breakdown (raw data, per `01_curve_preprocessing_v6.py`)

Same layout as the table above, but each chip now gets 2 sub-columns: **Active**
(pixels that survived `01`'s own lacewing + non-temperature + gain + linearity-fit
gates -- `Well.idx_active`, exactly what feeds `curve_for_training.joblib`) and
**Inactive** (every other pixel in that well's slice of the sensor grid), each shown
as count and % of that well's total pixel count. Loaded directly from the raw
readout files via `01_curve_preprocessing_v6.py`'s own `load_and_preprocess_v6` --
not re-derived or estimated -- so this doubles as a consistency check against the
active-pixel counts (`n_curves`) in the table above.


In [7]:
from pathlib import Path

# Only chip_v6_utils.load_and_preprocess_v6 is actually needed here -- import it
# directly (abl6-style: sys.path.insert with absolute paths) instead of execing all of
# 01_curve_preprocessing_v6.py (which drags in titan_v4/sigmoid_fitting for nothing).
sys.path.insert(0, str(Path(_MAIN_DIR) / "utils" / "01_curve_preprocessing"))

# chip_v6_utils.py's own titan_v6 import is CWD-relative ("../../titan_v6") -- chdir to
# main/ so it resolves, then chdir back. Must restore: _MAIN_DIR (cell 1) is itself
# derived from os.getcwd(), so a stray leftover chdir would corrupt it on a later re-run.
_orig_cwd = os.getcwd()
os.chdir(_MAIN_DIR)
try:
    from chip_v6_utils import load_and_preprocess_v6
finally:
    os.chdir(_orig_cwd)


def load_pixel_activity_counts(dataset_name):
    """Raw (n_active, n_total) pixel counts per well straight from 01's own
    pixel-activity pipeline (chip_v6_utils.load_and_preprocess_v6 -> Well.idx_active),
    summed across every vref slice (usually just one)."""
    exp_path = Path(EXP_FOLDER) / dataset_name
    all_exp_data = load_and_preprocess_v6(
        exp_path, n_wells=config.N_WELLS, n_a_type=config.N_A_TYPE, vref_ref_idx="all")

    rows = []
    for well_idx in range(config.N_WELLS):
        n_active, n_total = 0, 0
        for exp in all_exp_data:
            idx_active = np.asarray(exp.wells_list[well_idx].idx_active)
            n_active += int(idx_active.sum())
            n_total += len(idx_active)
        rows.append((well_idx, n_active, n_total))
    return rows


activity_records = []
for name in DATASETS:
    print(f"\n--- {name} ---")
    for well_idx, n_active, n_total in load_pixel_activity_counts(name):
        activity_records.append({
            "chip": chip_label(name),
            "well": well_idx,
            "label": config.LABEL_MAPPINGS.get(name, {}).get(well_idx),
            "concentration": config.CONC_MAPPINGS.get(name, {}).get(well_idx),
            "n_active": n_active,
            "n_total": n_total,
        })

activity_df = pd.DataFrame.from_records(activity_records)

# Consistency check: n_active here should exactly match n_curves in well_df, since
# both trace back to the same Well.idx_active selection.
_check = well_df.merge(activity_df, on=["chip", "well"], suffixes=("", "_raw"))
_mismatch = _check[_check["n_curves"] != _check["n_active"]]
if _mismatch.empty:
    print("\n[OK] n_active matches curve_for_training.joblib's n_curves for every well.")
else:
    print("\n[!] Mismatch vs. curve_for_training.joblib for:")
    display(_mismatch[["chip", "well", "label", "n_curves", "n_active"]])

activity_df



--- D20260806_E00_C00_F4500KHz_U_DDM_01_06 ---
[INFO] D20260806_E00_C00_F4500KHz_U_DDM_01_06: found 1 readout file(s). Processing...
Loaded vref sweep from 20260806T111947.79_vref_sweep.bin: n_vrefs=1
[INFO] Slice 1/1: DDM_01_06__vref_idx=0_vref=2730
EXP PATH /vol/bitbucket/gk225/POC_DDM_datasets/POC_DDM_final/D20260806_E00_C00_F4500KHz_U_DDM_01_06 
VREF PATH /vol/bitbucket/gk225/POC_DDM_datasets/POC_DDM_final/D20260806_E00_C00_F4500KHz_U_DDM_01_06/20260806T111947.79_vref_sweep.bin 
Load vref... 
Loaded vref list: n_vrefs=1, first=2730, last=2730
EXP PATH /vol/bitbucket/gk225/POC_DDM_datasets/POC_DDM_final/D20260806_E00_C00_F4500KHz_U_DDM_01_06 
READOUT PATH /vol/bitbucket/gk225/POC_DDM_datasets/POC_DDM_final/D20260806_E00_C00_F4500KHz_U_DDM_01_06/20260806T112024.90_readout_time.bin 
Load data... 


(290, 204, 1, 868)
Time vector: n=868, median_dt=3s, duration≈41.13min
(59160,)
new gain shape  (59160,)


/vol/bitbucket/gk225/POC_DDM/gk_code/main/../../titan_v6/linearise.py:48: RuntimeWarning: divide by zero encountered in log
  log_div = np.log((tau_i - C) / A) / np.log((tau_f - C) / A)  # Intermediate operation
/vol/bitbucket/gk225/POC_DDM/gk_code/main/../../titan_v6/linearise.py:48: RuntimeWarning: invalid value encountered in divide
  log_div = np.log((tau_i - C) / A) / np.log((tau_f - C) / A)  # Intermediate operation
/vol/bitbucket/gk225/POC_DDM/gk_code/main/../../titan_v6/linearise.py:49: RuntimeWarning: divide by zero encountered in divide
  D = (V_i - (log_div * V_f)) / (1 - log_div)  # Calculate D
/vol/bitbucket/gk225/POC_DDM/gk_code/main/../../titan_v6/linearise.py:50: RuntimeWarning: divide by zero encountered in log
  B = - (1 / (V_f - D)) * np.log((tau_f - C) / A)  # Calculate B


(290, 204, 1, 868)
Data loaded. 
Preprocessing start...
Active pixel counts (ref_idx=0, vref=2730.0): lacewing=19807, non_temp=56782, gain=17255, lin=17254, combined=17254
v06 temperature trace from readout meta: n=868, finite=868, min=0, max=452
v06 temperature: no usable temp_log.bin found, plotting readout phase only (868 samples).
Preprocessing end.
[OK] Completed slice 1/1 (ref_idx=0).

--- D20260807_E00_C00_F4500KHz_U_DDM_02_07 ---
[INFO] D20260807_E00_C00_F4500KHz_U_DDM_02_07: found 1 readout file(s). Processing...
Loaded vref sweep from 20260807T131525.38_vref_sweep.bin: n_vrefs=1
[INFO] Slice 1/1: DDM_02_07__vref_idx=0_vref=2730
EXP PATH /vol/bitbucket/gk225/POC_DDM_datasets/POC_DDM_final/D20260807_E00_C00_F4500KHz_U_DDM_02_07 
VREF PATH /vol/bitbucket/gk225/POC_DDM_datasets/POC_DDM_final/D20260807_E00_C00_F4500KHz_U_DDM_02_07/20260807T131525.38_vref_sweep.bin 
Load vref... 
Loaded vref list: n_vrefs=1, first=2730, last=2730
EXP PATH /vol/bitbucket/gk225/POC_DDM_datasets/POC_D

/vol/bitbucket/gk225/POC_DDM/gk_code/main/../../titan_v6/linearise.py:48: RuntimeWarning: invalid value encountered in log
  log_div = np.log((tau_i - C) / A) / np.log((tau_f - C) / A)  # Intermediate operation
/vol/bitbucket/gk225/POC_DDM/gk_code/main/../../titan_v6/linearise.py:50: RuntimeWarning: invalid value encountered in log
  B = - (1 / (V_f - D)) * np.log((tau_f - C) / A)  # Calculate B


(290, 204, 2, 437)
Data loaded. 
Preprocessing start...
Active pixel counts (ref_idx=0, vref=3410.0): lacewing=19602, non_temp=56782, gain=129, lin=108, combined=102
v06 temperature trace from readout meta: n=437, finite=437, min=0, max=489
v06 temperature: no usable temp_log.bin found, plotting readout phase only (437 samples).
Preprocessing end.
[OK] Completed slice 1/2 (ref_idx=0).
[INFO] Slice 2/2: DDM_03_01__vref_idx=1_vref=3510
EXP PATH /vol/bitbucket/gk225/POC_DDM_datasets/POC_DDM_final/D20260808_E00_C00_F4500KHz_U_DDM_03_01 
VREF PATH /vol/bitbucket/gk225/POC_DDM_datasets/POC_DDM_final/D20260808_E00_C00_F4500KHz_U_DDM_03_01/20260808T162013.42_vref_sweep.bin 
Load vref... 
Loaded vref list: n_vrefs=2, first=3410, last=3510
EXP PATH /vol/bitbucket/gk225/POC_DDM_datasets/POC_DDM_final/D20260808_E00_C00_F4500KHz_U_DDM_03_01 
READOUT PATH /vol/bitbucket/gk225/POC_DDM_datasets/POC_DDM_final/D20260808_E00_C00_F4500KHz_U_DDM_03_01/20260808T162124.38_readout_time.bin 
Load data... 
(290

,chip,well,label,concentration,n_active,n_total
0,DDM_01,0,IAV,1000000,1970,5916
1,DDM_01,1,IAV,100000,1949,5916
2,DDM_01,2,IAV,10000,1858,5916
3,DDM_01,3,IBV,10000,1907,5916
4,DDM_01,4,Kp,100000,1921,5916
5,DDM_01,5,Cov,1000000,1848,5916
6,DDM_01,6,Hadv,1000000,1919,5916
7,DDM_01,7,Hadv,100000,0,5916
8,DDM_01,8,PC,0,791,5916
9,DDM_01,9,NC-ALL,0,1135,5916


In [8]:
def build_activity_profile_table(activity_df):
    df = activity_df.copy()
    df["target"] = df["label"].map(TARGET_DISPLAY_NAME)

    target_rows = df[df["target"].notna()]
    col_order = pd.MultiIndex.from_product([TARGET_ORDER, CONC_ORDER], names=["target", "concentration"])

    def _wide(value_col):
        w = target_rows.pivot_table(index="chip", columns=["target", "concentration"],
                                    values=value_col, aggfunc="sum", fill_value=0)
        w = w.reindex(columns=col_order, fill_value=0)
        pc = df[df["label"] == "PC"].groupby("chip")[value_col].sum()
        nc = df[df["label"] == "NC-ALL"].groupby("chip")[value_col].sum()
        w[("PC", "")] = pc.reindex(w.index, fill_value=0)
        w[("NC", "")] = nc.reindex(w.index, fill_value=0)
        w[("Total", "")] = w.sum(axis=1)
        w = w.reindex(index=[chip_label(n) for n in DATASETS])
        w.columns = pd.MultiIndex.from_tuples(w.columns, names=["target", "concentration"])
        return w

    return _wide("n_active"), _wide("n_total")


active_profile, total_profile = build_activity_profile_table(activity_df)
active_profile


target          IAV                  IBV                   KP                 \
concentration 10000 100000 1000000 10000 100000 1000000 10000 100000 1000000   
chip                                                                           
DDM_01         1858   1949    1970  1907      0       0     0   1921       0   
DDM_02            0   1855    1816  1469   1799    1047  1911      0       0   
DDM_03         1786      0       0     0   1852    1903  1960   1663    1964   
DDM_04            0      0    1920  1921      0       0     0   1884    1950   
DDM_05            0   1955       0     0      0    1919  1943      0       0   
DDM_06         1890      0       0  1827   1936       0     0      0    1864   

target        SARS-CoV-2                 hAdV                   PC    NC  \
concentration      10000 100000 1000000 10000 100000 1000000               
chip                                                                       
DDM_01                 0      0    1848     0      0    1919   791  1135   
DDM_02                 0   1851       0  1894      0       0  1814  1778   
DDM_03              1928      0       0     0      0    1828  1755  1642   
DDM_04              1920   1913    1808     0   1943       0  1861  1315   
DDM_05                 0   1921    1884  1903   1945    1979  1896  1901   
DDM_06              1941   1916       0  3787      0       0  1715  1810   

target         Total  
concentration         
chip                  
DDM_01         15298  
DDM_02         17234  
DDM_03         18281  
DDM_04         18435  
DDM_05         19246  
DDM_06         18686

In [9]:
def to_latex_activity_table(active_profile, total_profile, caption, label, excluded_cells=None):
    excluded_cells = excluded_cells or set()
    chips = list(active_profile.index)
    n_chips = len(chips)

    def get(profile, chip, col):
        return profile.loc[chip, col] if col in profile.columns else 0

    def fmt(chip, row_label, conc):
        a = int(get(active_profile, chip, (row_label, conc)))
        tot = int(get(total_profile, chip, (row_label, conc)))
        inact = tot - a
        pct_a = (a / tot * 100) if tot else 0.0
        pct_i = (inact / tot * 100) if tot else 0.0
        s_a, s_i = f"{a:,} ({pct_a:.1f}\\%)", f"{inact:,} ({pct_i:.1f}\\%)"
        if (chip, row_label, conc) in excluded_cells:
            s_a, s_i = f"\\sout{{{s_a}}}", f"\\sout{{{s_i}}}"
        return s_a, s_i

    col_spec = "ll" + "rr" * n_chips
    lines = []
    lines.append("\\begin{table}[htbp]")
    lines.append("    \\centering")
    lines.append(f"    \\caption{{{caption}}}")
    lines.append(f"    \\label{{{label}}}")
    lines.append("    \\small")
    lines.append(f"    \\begin{{tabular}}{{@{{}}{col_spec}@{{}}}}")
    lines.append("    \\toprule")

    chip_header = ["", ""] + [f"\\multicolumn{{2}}{{c}}{{\\textbf{{{c}}}}}" for c in chips]
    lines.append("    " + " & ".join(chip_header) + " \\\\")
    lines.append("    " + " ".join(f"\\cmidrule(lr){{{3+2*i}-{4+2*i}}}" for i in range(n_chips)))
    sub_header = ["\\textbf{Target}", "\\textbf{Conc.}"] + ["\\textbf{Active}", "\\textbf{Inactive}"] * n_chips
    lines.append("    " + " & ".join(sub_header) + " \\\\")
    lines.append("    \\midrule")

    for t in TARGET_ORDER:
        for i, c in enumerate(CONC_ORDER):
            target_cell = f"\\multirow{{{len(CONC_ORDER)}}}{{*}}{{{t}}}" if i == 0 else ""
            cells = [target_cell, CONC_DISPLAY[c]]
            for chip in chips:
                cells.extend(fmt(chip, t, c))
            lines.append("    " + " & ".join(cells) + " \\\\")
        lines.append("    \\midrule")

    for extra in ["PC", "NC"]:
        cells = [f"\\multicolumn{{2}}{{l}}{{\\textbf{{{extra}}}}}"]
        for chip in chips:
            cells.extend(fmt(chip, extra, ""))
        lines.append("    " + " & ".join(cells) + " \\\\")

    lines.append("    \\midrule")
    cells = ["\\multicolumn{2}{l}{\\textbf{Total}}"]
    for chip in chips:
        cells.extend(fmt(chip, "Total", ""))
    lines.append("    " + " & ".join(cells) + " \\\\")

    lines.append("    \\bottomrule")
    lines.append("    \\end{tabular}")
    lines.append("\\end{table}")
    return "\n".join(lines)


activity_latex_table = to_latex_activity_table(
    active_profile, total_profile,
    caption="Active vs.\\ non-active pixel counts across the six Lacewing eLAMP chips in the final\\_6\\_chip\\_clean\\_nn LOFO group, by target and concentration (copies/reaction), including each chip's positive (PC) and negative (NC) control wells. Struck-through values are excluded from LOFO training for that chip.",
    label="tab:final_6chip_pixel_activity",
    excluded_cells=excluded_cells,
)
print(activity_latex_table)


\begin{table}[htbp]
    \centering
    \caption{Active vs.\ non-active pixel counts across the six Lacewing eLAMP chips in the final\_6\_chip\_clean\_nn LOFO group, by target and concentration (copies/reaction), including each chip's positive (PC) and negative (NC) control wells. Struck-through values are excluded from LOFO training for that chip.}
    \label{tab:final_6chip_pixel_activity}
    \small
    \begin{tabular}{@{}llrrrrrrrrrrrr@{}}
    \toprule
     &  & \multicolumn{2}{c}{\textbf{DDM_01}} & \multicolumn{2}{c}{\textbf{DDM_02}} & \multicolumn{2}{c}{\textbf{DDM_03}} & \multicolumn{2}{c}{\textbf{DDM_04}} & \multicolumn{2}{c}{\textbf{DDM_05}} & \multicolumn{2}{c}{\textbf{DDM_06}} \\
    \cmidrule(lr){3-4} \cmidrule(lr){5-6} \cmidrule(lr){7-8} \cmidrule(lr){9-10} \cmidrule(lr){11-12} \cmidrule(lr){13-14}
    \textbf{Target} & \textbf{Conc.} & \textbf{Active} & \textbf{Inactive} & \textbf{Active} & \textbf{Inactive} & \textbf{Active} & \textbf{Inactive} & \textbf{Active} & \textbf

In [10]:
activity_out_path = os.path.join(_MAIN_DIR, "notebooks", "final_6chip_pixel_activity_table.tex")
with open(activity_out_path, "w") as f:
    f.write(activity_latex_table + "\n")
print(f"Saved -> {activity_out_path}")


Saved -> /vol/bitbucket/gk225/POC_DDM/gk_code/main/notebooks/final_6chip_pixel_activity_table.tex
